# **Lecture 25: Statistical Methods - Advanced Monte Carlo Methods**

**Unit:** 3 (Statistical Methods)

**Topic:** Probability Theory, High-Dimensional Integration, and Random Walks

***

## **1. The Formal Mathematics of Monte Carlo**

Monte Carlo methods rely on the **Law of Large Numbers** and the **Central Limit Theorem** to evaluate deterministic problems using random sampling. 

Suppose we want to evaluate a multi-dimensional definite integral over a volume $V$:
$$I = \int_{V} f(\vec{x}) d\vec{x}$$

If we sample $N$ uniformly distributed random points $\vec{x}_i$ within the volume $V$, the Fundamental Theorem of Monte Carlo Integration states that the integral can be approximated by the expected value (the mean) of the function evaluated at those points:
$$I \approx I_N = V \langle f \rangle = \frac{V}{N} \sum_{i=1}^{N} f(\vec{x}_i)$$

### **1.1 Why Monte Carlo? The Curse of Dimensionality**
Why use random points instead of a structured grid like Simpson's Rule? The answer lies in the **Variance** and **Standard Error**.

The variance of our estimator $I_N$ is given by:
$$Var(I_N) = \frac{V^2}{N} Var(f) = \frac{V^2}{N} \left( \frac{1}{N} \sum_{i=1}^{N} f^2(\vec{x}_i) - \langle f \rangle^2 \right)$$

The standard error (uncertainty) is the square root of the variance:
$$\sigma_I \propto \frac{1}{\sqrt{N}}$$

**Crucial Insight:** The error of Monte Carlo integration decreases as $O(N^{-1/2})$, **regardless of the number of dimensions $d$**. 
Conversely, the error of a standard grid-based method (like the Trapezoidal rule) scales as $O(N^{-2/d})$. If you are simulating a complex 6-dimensional phase space ($d=6$), the grid error scales as $O(N^{-1/3})$, which is worse than Monte Carlo. For high-dimensional astrophysical problems, Monte Carlo is the only computationally viable option.

## **2. Hit-or-Miss Integration (Indicator Functions)**

A special case of Monte Carlo integration is finding the volume of a complex shape. We define a bounding box $V_{box}$ and an **Indicator Function** $I(\vec{x})$:
$$I(\vec{x}) = \begin{cases} 1 & \text{if } \vec{x} \text{ is inside the shape} \\ 0 & \text{if } \vec{x} \text{ is outside the shape} \end{cases}$$

The volume of the shape is exactly:
$$V_{shape} = \int_{V_{box}} I(\vec{x}) d\vec{x} \approx V_{box} \frac{N_{hits}}{N_{total}}$$

Let us apply this mathematical formulation to estimate the volume of a 3D sphere.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Define Parameters
N_total = 200000       # N points
R = 2.0                # Radius of the sphere

# 2. Uniform Random Sampling in a 3D Box (Domain: [-R, R] for x,y,z)
x = np.random.uniform(-R, R, N_total)
y = np.random.uniform(-R, R, N_total)
z = np.random.uniform(-R, R, N_total)

# 3. Evaluate the Indicator Function
indicator = (x**2 + y**2 + z**2) <= R**2
N_hits = np.sum(indicator)

# 4. Calculate Volumes based on the Fundamental Theorem
V_box = (2 * R)**3
V_mc = V_box * (N_hits / N_total)
V_analytical = (4.0 / 3.0) * np.pi * R**3

print(f"--- Monte Carlo Volume Estimation ---")
print(f"Samples (N): {N_total}")
print(f"Monte Carlo Estimate: {V_mc:.5f}")
print(f"Analytical Value:     {V_analytical:.5f}")

# 5. Calculate the Statistical Standard Error (sigma)
# For a binomial hit/miss process, the variance of the hit probability p is p*(1-p)/N
p = N_hits / N_total
variance_p = (p * (1 - p)) / N_total
standard_error = V_box * np.sqrt(variance_p)

print(f"Statistical Standard Error: +/- {standard_error:.5f}")
print(f"Actual True Error:          {abs(V_mc - V_analytical):.5f}")


## **3. The Physics of Radiative Transfer (Random Walks)**

Inside a star, the density is so high that photons cannot travel in straight lines. They scatter off electrons and ions. This is modeled mathematically as a **Random Walk**. 

Let $\vec{r}_i$ be the vector of the $i$-th step, with a constant mean free path length $l = |\vec{r}_i|$. The direction is completely random.
After $N$ steps, the total displacement vector $\vec{R}$ is:
$$\vec{R} = \sum_{i=1}^{N} \vec{r}_i$$

What is the expected distance the photon travels? Because directions are random, the expected vector is zero: $\langle \vec{R} \rangle = 0$. 

To find the actual distance from the origin, we must calculate the **Root Mean Square (RMS)** distance. We square the displacement vector:
$$\vec{R} \cdot \vec{R} = \left( \sum_{i=1}^{N} \vec{r}_i \right) \cdot \left( \sum_{j=1}^{N} \vec{r}_j \right)$$
$$\vec{R}^2 = \sum_{i=1}^{N} \vec{r}_i^2 + 2 \sum_{i=1}^{N} \sum_{j=i+1}^{N} \vec{r}_i \cdot \vec{r}_j$$

Because the scattering angles are completely uncorrelated, the dot product of any two different steps averages to zero: $\langle \vec{r}_i \cdot \vec{r}_j \rangle = 0$. 
We are left only with the diagonal terms (where $i = j$):
$$\langle \vec{R}^2 \rangle = \sum_{i=1}^{N} l^2 = N l^2$$

Therefore, the RMS distance traveled after $N$ steps is:
$$R_{rms} = l \sqrt{N}$$

### **Astrophysical Consequence: Photon Escape Time**
If a star has a radius $R_{star}$ and a photon mean free path $l$, the number of steps required to escape is $N = (R_{star} / l)^2$. 
The total physical distance the photon travels is $D_{total} = N \cdot l$. 
The time it takes to escape is $t = D_{total} / c$, giving us the famous diffusion time equation:
$$t_{escape} \approx \frac{R_{star}^2}{l \cdot c}$$
For the Sun, $R_{star} = 7 \times 10^8$ m and $l \approx 0.001$ m. This math proves it takes a photon approximately $170,000$ years to escape the Sun!

In [ ]:
# Simulate a 2D Photon Random Walk
N_steps = 5000
l_path = 1.0 # Mean free path length

x_path = np.zeros(N_steps)
y_path = np.zeros(N_steps)

x_current, y_current = 0.0, 0.0

for i in range(1, N_steps):
    # Uniformly random scattering angle theta [0, 2pi)
    theta = np.random.uniform(0, 2 * np.pi)
    
    x_current += l_path * np.cos(theta)
    y_current += l_path * np.sin(theta)
    
    x_path[i] = x_current
    y_path[i] = y_current

# Calculate final metrics
final_distance = np.sqrt(x_path[-1]**2 + y_path[-1]**2)
theoretical_rms = l_path * np.sqrt(N_steps)

print(f"Total Steps Taken: {N_steps}")
print(f"Theoretical RMS Distance: {theoretical_rms:.2f}")
print(f"Actual Final Distance: {final_distance:.2f}")

# Visualization
plt.figure(figsize=(9, 9))
plt.plot(x_path, y_path, 'k-', alpha=0.4, lw=0.8, label='Photon Path')
plt.plot(0, 0, 'go', markersize=10, label='Core (Origin)')
plt.plot(x_path[-1], y_path[-1], 'ro', markersize=8, label='Final Position')

# Draw the theoretical RMS radius circle
circle = plt.Circle((0, 0), theoretical_rms, color='blue', fill=False, linestyle='--', lw=2, label=r'Theoretical $R_{rms} = l\sqrt{N}$')
plt.gca().add_patch(circle)

plt.title("Astrophysical Random Walk (Radiative Transfer)")
plt.xlabel("X Position")
plt.ylabel("Y Position")
plt.legend(loc='upper right')
plt.grid(True, linestyle=':')
plt.axis('equal')
plt.show()


## **4. Student Exercises**

### **Exercise 1: Proving the $1/\sqrt{N}$ Error Scaling**
In Section 1, we proved mathematically that the error of Monte Carlo integration scales inversely with the square root of $N$. Let's prove it computationally.
**Task:** 
1. Write a `for` loop that iterates through an array of sample sizes: $N = [10^2, 10^3, 10^4, 10^5, 10^6]$.
2. For each $N$, estimate the value of $\pi$ using the 2D Hit-or-Miss method (circle inside a square).
3. Calculate the absolute error $|\pi_{estimated} - \pi_{true}|$.
4. Plot the Error on the Y-axis and $N$ on the X-axis using a log-log plot (`plt.loglog`). 
5. Overlay a reference line $y = 1/\sqrt{x}$. The slopes should match perfectly.

---

### **Exercise 2: Optical Depth and Escape Probability**
In astrophysics, **Optical Depth ($\tau$)** is defined as the number of mean free paths through a medium: $\tau = R_{star} / l$. 
The probability that a photon escapes a medium of optical depth $\tau$ without scattering even once is given by $P = e^{-\tau}$.

**Task:** 
Set the star radius to $R = 5.0$ and mean free path $l = 1.0$ (meaning $\tau = 5$). Simulate $10,000$ individual photons. Track how many photons escape the circle of radius $R$ in exactly **1 step** (no scattering). Compare this empirical probability to the theoretical $e^{-\tau}$.